In [ ]:
%load_ext autoreload
%autoreload 2

In [21]:
data = pd.read_csv(conf['prep_text']['prep_fn'])
data['ttp'] = data['ttp'].map(lambda x: eval(x))


In [22]:

data.loc[(data['ttp'].map(lambda x: len(x)==0)), 'split']


13564     tr
13565     tr
13566    val
13567    val
13569    val
        ... 
29294     tr
29295     tr
29296     tr
29297     tr
29298     tr
Name: split, Length: 12030, dtype: object

___

In [8]:
# DN = 'C://work/dev/python/progs/texts/sec_bert/'
DN = '/home/jovyan/work/sec_bert/'

import os
os.chdir(DN)

In [9]:
from sklearn.metrics import (average_precision_score, log_loss, confusion_matrix,
                            precision_recall_fscore_support, f1_score)

import matplotlib.pyplot as plt


import os

from ruamel.yaml import YAML
import pandas as pd
import numpy as np
import joblib

from collections import defaultdict
from itertools import chain

import click
import json

import torch
import torch.nn as nn
from torch.optim.lr_scheduler import ExponentialLR, MultiStepLR
from torch.utils.data import DataLoader, Dataset


from transformers import BertTokenizer, BertForSequenceClassification
from transformers import AutoTokenizer, AutoModelForMaskedLM, BertConfig, AutoModel
from transformers import DataCollatorWithPadding
from transformers import RobertaTokenizer, RobertaModel

DEVICE = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

import sys
sys.path.append('.')
from src.funcs import set_seed
from src.funcs import metric_multi
from src.funcs import get_opt_thresh, get_preds
from src.funcs import get_conf_df
from src.spec_nn_funcs import TextDFDataset, TextModelClass
from ruamel.yaml import YAML

/opt/conda/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB

from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import RobustScaler

from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import RobustScaler, StandardScaler

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.tree import DecisionTreeClassifier

In [11]:


conf = YAML().load(open('params.yaml'))
conf_ttp = YAML().load(open('dvc_pipes/ttp/params_ttp.yaml'))
conf_bert = YAML().load(open('dvc_pipes/bert/params_bert.yaml'))
conf_bert_ttp = YAML().load(open('dvc_pipes/bert_ttp/params_bert_ttp.yaml'))

set_seed(conf['seed'])

In [ ]:
bert_type = conf_bert['nn_bert']['bert_type']


In [ ]:
VALID_BATCH_SIZE = conf_bert['nn']['batch_size']
TRAIN_BATCH_SIZE = conf_bert['nn']['batch_size']
MAX_SEQ_LENGTH = conf_bert['nn']['maxlen']

checkpoint = 'data/external/models/SecureBERT_Plus/snapshots/4c48ccdb8d2019f179b07dfa27656c655394d78e'
tokenizer = RobertaTokenizer.from_pretrained(checkpoint)
tokenizer_opts = {'max_length':MAX_SEQ_LENGTH, 'return_tensors':"pt", 'padding':True, 'truncation':True, 'add_special_tokens':True}


# Загрузка данных

In [ ]:
mlb_ttp = joblib.load(conf['prep_text']['ttp_mlb_fn'])


In [ ]:
data_ttp = pd.read_csv(conf_ttp['feat_gen_ttp']['data_fn'])
data_ttp['target'] = data_ttp['target'].map(lambda x: eval(x))
data_ttp['ttp'] = data_ttp['ttp'].map(lambda x: eval(x))



In [ ]:
model_bert_ttp = torch.load(f'{conf_bert_ttp["nn_bert_ttp"]["model_fn"]}')



In [ ]:
feat_ttp = pd.read_csv(conf_ttp['feat_eng_ttp']['feat_final_fn'])



In [ ]:
tr_ttp_ds = TextDFDataset(data_ttp.query('split=="tr"').reset_index(drop=True), tokenizer=tokenizer, tokenizer_opts=tokenizer_opts)
val_ttp_ds = TextDFDataset(data_ttp.query('split=="val"').reset_index(drop=True), tokenizer=tokenizer, tokenizer_opts=tokenizer_opts)
ts_ttp_ds = TextDFDataset(data_ttp.query('split=="ts"').reset_index(drop=True), tokenizer=tokenizer, tokenizer_opts=tokenizer_opts)

tr_ttp_ld = DataLoader(tr_ttp_ds, batch_size = TRAIN_BATCH_SIZE, shuffle = False, collate_fn = DataCollatorWithPadding(tokenizer=tokenizer))
val_ttp_ld = DataLoader(val_ttp_ds, batch_size = VALID_BATCH_SIZE, shuffle = False, collate_fn = DataCollatorWithPadding(tokenizer=tokenizer))
ts_ttp_ld = DataLoader(ts_ttp_ds, batch_size = VALID_BATCH_SIZE, shuffle = False, collate_fn = DataCollatorWithPadding(tokenizer=tokenizer))


In [ ]:
Y_ttp_val_proba = np.array(get_preds(model_bert_ttp, ld=val_ttp_ld)['pred'])
Y_ttp_tr_proba = np.array(get_preds(model_bert_ttp, ld=tr_ttp_ld)['pred'])
Y_ttp_ts_proba = np.array(get_preds(model_bert_ttp, ld=ts_ttp_ld)['pred'])

In [ ]:
thresh_ttp_l = get_opt_thresh(y_true = np.array(data_ttp.loc[data_ttp.split=='tr', 'target'].values.tolist()), 
                          probas = Y_ttp_tr_proba, mlb = mlb_ttp, 
                          opt_metric=conf['train_eval_model']['opt_metric'], 
                          thresh_space_l=np.arange(0.001, 1, 0.002), dump_fn = None)


In [ ]:
ttp_df = pd.concat([data_ttp[['sentence', 'ttp', 'labels',	'url', 'target', 'split']].query('split=="tr"').assign(proba_ttp = Y_ttp_tr_proba.tolist()),
          data_ttp[['sentence', 'ttp', 'labels',	'url', 'target', 'split']].query('split=="val"').assign(proba_ttp = Y_ttp_val_proba.tolist()),
           data_ttp[['sentence', 'ttp', 'labels',	'url', 'target', 'split']].query('split=="ts"').assign(proba_ttp = Y_ttp_ts_proba.tolist())
          ], axis=0, ignore_index=True)

ttp_df['pred_ttp'] = ttp_df['proba_ttp'].map(lambda x: [int(val>=thresh) for val, thresh in zip(x, thresh_ttp_l)])

ttp_df['pred_str_ttp'] = ttp_df['pred_ttp'].map(lambda x: mlb_ttp.inverse_transform(np.array([x]))[0])


In [ ]:
p_val_macro, r_val_macro, f1_val_macro, sup = precision_recall_fscore_support(np.array(ttp_df.query('split in ["val", "ts"]')['target'].values.tolist()), 
                                                    np.array(ttp_df.query('split in ["val", "ts"]')['pred_ttp'].values.tolist()), average='macro')

f1_val_macro

In [ ]:
def add_cols(x):
    ttp_idx = mlb_ttp.transform([x['ttp']]).argmax()
    x['prob_real'] = x['proba_ttp'][ttp_idx]
    x['prob_need'] = thresh_ttp_l[ttp_idx]
    return x

ttp_df = ttp_df.apply(add_cols, axis=1)
ttp_df = data_ttp.drop(columns='index').reset_index()[['index', 'sentence']].merge(ttp_df, on='sentence')
ttp_df = ttp_df.set_index('index')

tr_sel = ttp_df.split=='tr'
val_sel = ttp_df.split=='val'

In [ ]:
f1, res_l = metric_multi(np.array(ttp_df.query('split in ["val", "ts"]')['target'].values.tolist()), np.array(ttp_df.query('split in ["val", "ts"]')['pred_ttp'].values.tolist()), f1_score)

N = 30
bad_df = pd.DataFrame({'qual':res_l, 'class':mlb_ttp.classes_}).sort_values(by='qual')
bad_cls_l = bad_df.head(N)['class'].tolist()

print(f1)
bad_df.head(2)

In [ ]:
ttp_df[(val_sel)&(ttp_df['ttp'].map(lambda x: any([it in x for it in bad_cls_l])))].explode('ttp')['ttp'].value_counts()

- одиночные классы в основном

In [ ]:
from src.interpret.class_interp import BayesExplain
from src.interpret.class_interp import class_word_stat

exp = BayesExplain(data=ttp_df, feat_data=feat_ttp, mlb=mlb_ttp).fit()


# качество только для частых (если предположить, что деградация качества из-за малой частоты отдельных классов)

In [ ]:
ttp_cut_df = ttp_df.copy()

In [ ]:
unfreq_thresh = 10
unfreq_ttp_l = ttp_df.query('split in ["val", "ts"]').explode('ttp').groupby('ttp').size().loc[lambda x: x<unfreq_thresh].index

In [ ]:
empty_sel = ttp_df['ttp'].map(lambda x: len(x)==0)

In [ ]:
# сделаем грубее, удалим тех, что стали пустыми, а до не были
ttp_cut_df['ttp'] = ttp_cut_df['ttp'].map(lambda x: [it for it in x if not it in unfreq_ttp_l])
empty_after_sel = ttp_cut_df['ttp'].map(lambda x: len(x)==0)

In [ ]:
ttp_cut_df['target'] = ttp_cut_df['ttp'].map(lambda x: mlb_ttp.transform([x]).squeeze())

In [ ]:
# примерно 10% данных убрали
ttp_cut_df = ttp_cut_df[~(empty_after_sel&(~empty_sel))]

In [ ]:
# качество еще упало, значит
p_val_macro, r_val_macro, f1_val_macro, sup = precision_recall_fscore_support(np.array(ttp_cut_df.query('split in ["val", "ts"]')['target'].values.tolist()), 
                                                    np.array(ttp_cut_df.query('split in ["val", "ts"]')['pred_ttp'].values.tolist()), average='macro')

f1_val_macro

In [ ]:
f1, res_l = metric_multi(np.array(ttp_cut_df.query('split in ["val", "ts"]')['target'].values.tolist()), np.array(ttp_cut_df.query('split in ["val", "ts"]')['pred_ttp'].values.tolist()), f1_score)

N = 30
bad_cut_df = pd.DataFrame({'qual':res_l, 'class':mlb_ttp.classes_}).sort_values(by='qual')
print(f1)
bad_cut_df.head(2)

## сравним

In [ ]:
'T1001.001' in unfreq_ttp_l

In [ ]:
bad_cut_df.merge(bad_df, on='class').query('qual_y>qual_x').head()

### на ряде малых потеряли качество

### улучшение на прежних малых с 0

In [ ]:
# все, где у старого лучше качество как раз маленькие 
[it for it in bad_cut_df.merge(bad_df, on='class').query('qual_y>qual_x')['class'].tolist() if not it in unfreq_ttp_l]

In [ ]:
# T1505.003, T1001.001
bad_cut_df.merge(bad_df, on='class').loc[lambda x: x['class']=="T1505.003"]

In [ ]:
bad_cut_df.merge(bad_df, on='class').loc[lambda x: x['class']=="T1001.001"]

In [ ]:
num = mlb_ttp.transform([["T1001.001"]]).argmax()

не предсказывали и не стало

In [ ]:
np.array(ttp_cut_df.query('split in ["val", "ts"]')['pred_ttp'].values.tolist())[:, num].sum()

In [ ]:
np.array(ttp_cut_df.query('split in ["val", "ts"]')['target'].values.tolist())[:, num].sum()

не предсказывали, но был

In [ ]:
np.array(ttp_df.query('split in ["val", "ts"]')['pred_ttp'].values.tolist())[:, num].sum()

In [ ]:
np.array(ttp_df.query('split in ["val", "ts"]')['target'].values.tolist())[:, num].sum()

In [ ]:
T1505.003

In [ ]:
num = mlb_ttp.transform([["T1505.003"]]).argmax()
num

In [ ]:
np.array(ttp_cut_df.query('split in ["val", "ts"]')['target'].values.tolist())[:, num].sum()

In [ ]:
ttp_cut_df[ttp_cut_df['target'].map(lambda x: x[num]==1)]

напредсказывали хотя не было

In [ ]:
np.array(ttp_cut_df.query('split in ["val", "ts"]')['pred_ttp'].values.tolist())[:, num].sum()

In [ ]:
np.array(ttp_cut_df.query('split in ["val", "ts"]')['target'].values.tolist())[:, num].sum()

напредсказывали, неправильно и был 1 раз

In [ ]:
np.array(ttp_df.query('split in ["val", "ts"]')['pred_ttp'].values.tolist())[:, num].sum()

In [ ]:
np.array(ttp_df.query('split in ["val", "ts"]')['target'].values.tolist())[:, num].sum()

In [ ]:
ttp_df.query('split in ["val", "ts"]')[ttp_df.query('split in ["val", "ts"]')['ttp'].map(lambda x: "T1505.003" in x)]

# качество старого берта на данных не митр

In [ ]:
ttp_old_df = ttp_df.copy()


In [ ]:
mitre_sel = ttp_old_df['url'].str.contains('https://attack.mitre.org', na=False)
other_nempty_sel = (~mitre_sel) & (ttp_old_df['url'].notna())

In [ ]:
ttp_old_df = ttp_old_df[~mitre_sel]

In [ ]:
f1, res_l = metric_multi(np.array(ttp_old_df.query('split in ["val", "ts"]')['target'].values.tolist()), np.array(ttp_old_df.query('split in ["val", "ts"]')['pred_ttp'].values.tolist()), f1_score)

f1

In [ ]:
# качество еще упало, значит
p_val_macro, r_val_macro, f1_val_macro, sup = precision_recall_fscore_support(np.array(ttp_old_df.query('split in ["val", "ts"]')['target'].values.tolist()), 
                                                    np.array(ttp_old_df.query('split in ["val", "ts"]')['pred_ttp'].values.tolist()), average='macro')

f1_val_macro

## проверка качества по классам, которые есть в ~mitre_sel
Так как если класса в реальности нет, то качество может непредсказуемо быть 0 (если были прогнозы хотя бы по 1 экземпляру) или 1

In [ ]:
ttp_old_df = ttp_df.copy()


In [ ]:
cls_ar = ttp_df.loc[~mitre_sel].explode('ttp')['ttp'].dropna().unique()

In [ ]:
idx_ar = mlb_ttp.transform([[it] for it in cls_ar])
idx_ar = idx_ar.nonzero()[1]

In [ ]:
f1, res_l = metric_multi(np.array(ttp_old_df[mitre_sel].query('split in ["val", "ts"]')['target'].values.tolist()), np.array(ttp_old_df[mitre_sel].query('split in ["val", "ts"]')['pred_ttp'].values.tolist()), f1_score)
bad_mitre_df = pd.DataFrame({'qual':res_l, 'class':mlb_ttp.classes_}).sort_values(by='qual')

f1, np.mean([it for i, it in enumerate(res_l) if i in list(idx_ar)])

In [ ]:
f1, res_l = metric_multi(np.array(ttp_old_df.query('split in ["val", "ts"]')['target'].values.tolist()), np.array(ttp_old_df.query('split in ["val", "ts"]')['pred_ttp'].values.tolist()), f1_score)
bad_df = pd.DataFrame({'qual':res_l, 'class':mlb_ttp.classes_}).sort_values(by='qual')

f1, np.mean([it for i, it in enumerate(res_l) if i in list(idx_ar)])

## mitre vs ~mitre_sel

In [ ]:
ttp_old_df = ttp_df.copy()


In [ ]:
mitre_sel = ttp_old_df['url'].str.contains('https://attack.mitre.org', na=False)


In [ ]:
f1, res_l = metric_multi(np.array(ttp_old_df[mitre_sel].query('split in ["val", "ts"]')['target'].values.tolist()), np.array(ttp_old_df[mitre_sel].query('split in ["val", "ts"]')['pred_ttp'].values.tolist()), f1_score)
bad_mitre_df = pd.DataFrame({'qual':res_l, 'class':mlb_ttp.classes_}).sort_values(by='qual')

f1

In [ ]:
f1, res_l = metric_multi(np.array(ttp_old_df[~mitre_sel].query('split in ["val", "ts"]')['target'].values.tolist()), np.array(ttp_old_df[~mitre_sel].query('split in ["val", "ts"]')['pred_ttp'].values.tolist()), f1_score)
bad_df = pd.DataFrame({'qual':res_l, 'class':mlb_ttp.classes_}).sort_values(by='qual')

f1

### когда ~mitre дает прирост

In [ ]:
bad_mitre_df.merge(bad_df, on='class').query('qual_y>qual_x').head()

In [ ]:
num = mlb_ttp.transform([["T1040"]]).argmax()
num

2 раза предсказывали и 1 раз попали

In [ ]:
np.array(ttp_old_df.query('split in ["val", "ts"]')['pred_ttp'].values.tolist())[:, num].sum()

In [ ]:
np.array(ttp_old_df.query('split in ["val", "ts"]')['target'].values.tolist())[:, num].sum()

In [ ]:
val_old = ttp_old_df.query('split in ["val", "ts"]')
val_old[val_old['ttp'].map(lambda x: 'T1040' in x)]

не предсказывали, но был

In [ ]:
np.array(ttp_df[mitre_sel].query('split in ["val", "ts"]')['pred_ttp'].values.tolist())[:, num].sum()

In [ ]:
np.array(ttp_df[mitre_sel].query('split in ["val", "ts"]')['target'].values.tolist())[:, num].sum()

Предсказывали, но не попали и сразу 0, так как предсказывали другое

In [ ]:
val_old = ttp_old_df[mitre_sel].query('split in ["val", "ts"]')
val_old[val_old['ttp'].map(lambda x: 'T1040' in x)]

### когда ~mitre дает убыток

In [ ]:
bad_mitre_df.merge(bad_df, on='class').query('qual_y<qual_x').sort_values(by='qual_y').head()

## случайный классификатор

In [ ]:
prob_pclass = np.array(ttp_df['target'].to_numpy().tolist()).mean(axis=0)

def gen_synth_proba(size, prob_pclass):
    ar_l = []
    for i in range(size[1]):
        ar_l.append(np.random.choice([0,1], size=(size[0],), p=[1-prob_pclass[i], prob_pclass[i]]).reshape(-1, 1))

    return np.hstack(ar_l)

In [ ]:
ttp_old_df.query('split in ["val", "ts"]').shape[0]

In [ ]:
preds_synth_val_ts = gen_synth_proba(size=(ttp_old_df.query('split in ["val", "ts"]').shape[0], len(mlb_ttp.classes_)), prob_pclass=prob_pclass)

p, r, f1_val_ts, _ = precision_recall_fscore_support(np.array(ttp_old_df.query('split in ["val", "ts"]')['target'].values.tolist())
                                                     , preds_synth_val_ts, average='macro')
f1_val_ts

# глюк tram

In [ ]:
tram_df = pd.read_json(conf['get_data']['tram_fn']).drop(columns='doc_title')

In [ ]:
tram_df[tram_df.sentence.str.contains('^\w+T\d+', regex=True)]

# T1001.001

In [ ]:
bad_cls_l[0]

In [ ]:
ttp = bad_cls_l[0]
ttp_df[(val_sel)&ttp_df['ttp'].map(lambda x: ttp in x)]

In [ ]:
it = 0

with pd.option_context('display.max_colwidth', 100):
    display(ttp_df[(val_sel)&(ttp_df['ttp'].map(lambda x: ttp in x))].iloc[5*it:5*it+5])
    

## предсказания

In [ ]:
(ttp_df['proba_ttp'].map(lambda x: x[mlb_ttp.transform([[ttp]]).argmax()])>0.049).mean()

## статистика по словам

In [ ]:
index = 31728

In [ ]:
ttp_df.iloc[index]['sentence']

In [ ]:
with pd.option_context('display.max_columns', 100):
    # display(exp.text_explain(idx[N]))
    display(exp.text_explain(index, exp_type='algo'))

In [ ]:
exp.cls_explain(ttp, k=10)

## близкие тексты

In [ ]:
from src.interpret.close_k_feat import select_k_neigh

comp_df, _ = select_k_neigh(ttp_df, feat_ttp, k=6, idx=[index], target_col='ttp')

In [ ]:
with pd.option_context('display.max_colwidth', 200):
    display(comp_df.iloc[0:])

<div class='alert alert-info'>
В целом текст похож, не хватило cut-off 
</div>

# T1497

In [ ]:
ttp = 'T1497'


In [ ]:
it = 0

with pd.option_context('display.max_colwidth', 100):
    display(ttp_df[(val_sel)&(ttp_df['ttp'].map(lambda x: ttp in x))].iloc[5*it:5*it+5])
    

## предсказания

In [ ]:
(ttp_df['proba_ttp'].map(lambda x: x[mlb_ttp.transform([[ttp]]).argmax()])>0.01).mean()

## статистика по словам

In [ ]:
index = 31640

In [ ]:
ttp_df.iloc[index]['sentence']

In [ ]:
with pd.option_context('display.max_columns', 100):
    # display(exp.text_explain(idx[N]))
    display(exp.text_explain(index, exp_type='algo'))

In [ ]:
exp.cls_explain(ttp, k=10)

## близкие тексты

In [ ]:
from src.interpret.close_k_feat import select_k_neigh

comp_df, _ = select_k_neigh(ttp_df, feat_ttp, k=6, idx=[index], target_col='ttp')

In [ ]:
with pd.option_context('display.max_colwidth', 200):
    display(comp_df.iloc[0:])

<div class='alert alert-info'>
В целом тексты похожие, но вероятности не хватает для классификации
    
- Для текста - 31059, в принципе чуть не хватило для перешагивания cut-off, есть похожие, но из других классов
- index 31640 похож на T1622
</div>

# T1203

In [ ]:
ttp = 'T1203'


In [ ]:
it = 1

with pd.option_context('display.max_colwidth', 100):
    display(ttp_df[(val_sel)&(ttp_df['ttp'].map(lambda x: ttp in x))].iloc[5*it:5*it+5])
    

## предсказания

In [ ]:
(ttp_df['proba_ttp'].map(lambda x: x[mlb_ttp.transform([[ttp]]).argmax()])>0.02).mean()

## статистика по словам

In [ ]:
index = 31933	

In [ ]:
ttp_df.iloc[index]['sentence']

In [ ]:
with pd.option_context('display.max_columns', 100):
    # display(exp.text_explain(idx[N]))
    display(exp.text_explain(index, exp_type='algo'))

In [ ]:
exp.cls_explain(ttp, k=10)

## близкие тексты

In [ ]:
from src.interpret.close_k_feat import select_k_neigh

comp_df, _ = select_k_neigh(ttp_df, feat_ttp, k=6, idx=[index], target_col='ttp')

In [ ]:
with pd.option_context('display.max_colwidth', 200):
    display(comp_df.iloc[1:])

<div class='alert alert-info'>
    
- Для текста - 31701, в принципе чуть не хватила для перешагивания cut-off, есть похожие, но из других классов
-  Для других аналогично
</div>

# T1071

In [ ]:
ttp = 'T1071'


In [ ]:
it = 0
with pd.option_context('display.max_colwidth', 100):
    display(ttp_df[(val_sel)&(ttp_df['ttp'].map(lambda x: ttp in x))].iloc[5*it:5*it+5])
    

## предсказания

In [ ]:
(ttp_df['proba_ttp'].map(lambda x: x[mlb_ttp.transform([[ttp]]).argmax()])>0.03	).mean()

## статистика по словам

In [ ]:
index = 31995

In [ ]:
ttp_df.iloc[index]['sentence']

In [ ]:
with pd.option_context('display.max_columns', 100):
    # display(exp.text_explain(idx[N]))
    display(exp.text_explain(index, exp_type='algo'))

In [ ]:
exp.cls_explain(ttp, k=10)

## близкие тексты

In [ ]:
from src.interpret.close_k_feat import select_k_neigh

comp_df, _ = select_k_neigh(ttp_df, feat_ttp, k=6, idx=[index], target_col='ttp')

In [ ]:
with pd.option_context('display.max_colwidth', 200):
    display(comp_df.iloc[1:])

<div class='alert alert-info'>T1071 тексты похожи (особенно index = 31995), но не прошли cut-off. Здесь, кажется много бы решило единообразное упоминание C&C (в отчетах) и C2 (у митра) - если это одно и то же
</div>

# T1072

In [ ]:
ttp = 'T1072'


In [ ]:
it = 0
with pd.option_context('display.max_colwidth', 100):
    display(ttp_df[(val_sel)&(ttp_df['ttp'].map(lambda x: ttp in x))].iloc[5*it:5*it+5])
    

## предсказания

In [ ]:
(ttp_df['proba_ttp'].map(lambda x: x[mlb_ttp.transform([[ttp]]).argmax()])>0.000145		).mean()

## статистика по словам

In [ ]:
index = 29284

In [ ]:
ttp_df.iloc[index]['sentence']

In [ ]:
with pd.option_context('display.max_columns', 100):
    # display(exp.text_explain(idx[N]))
    display(exp.text_explain(index, exp_type='algo'))

In [ ]:
exp.cls_explain(ttp, k=10)

## близкие тексты

In [ ]:
from src.interpret.close_k_feat import select_k_neigh

comp_df, _ = select_k_neigh(ttp_df, feat_ttp, k=6, idx=[index], target_col='ttp')

In [ ]:
with pd.option_context('display.max_colwidth', 200):
    display(comp_df.iloc[1:])

<div class='alert alert-info'>Для митр T1072 и других иных источников тексты совсем разные
</div>